In [11]:
import pandas as pd
import numpy as np
import os
 
RAW_FILE = "data/CommMktArrivals2012.xls"
OUT_FILE = "data/cleaned_data.csv"

In [12]:
print("=" * 55)
print("STEP 1 : Loading & Preprocessing")
print("=" * 55)
 
df = pd.read_excel(RAW_FILE, engine="xlrd")
print(f"\n✅ Loaded raw data  →  {df.shape[0]} rows, {df.shape[1]} columns")

STEP 1 : Loading & Preprocessing

✅ Loaded raw data  →  21422 rows, 10 columns


In [13]:
str_cols = df.select_dtypes(include="object").columns.tolist() + \
           [c for c in df.columns if df[c].dtype.name == "string"]
 
for col in df.columns:
    try:
        df[col] = df[col].str.strip()
    except AttributeError:
        pass
 
print("✅ Stripped whitespace from all text columns")

✅ Stripped whitespace from all text columns


In [14]:
drop_cols = ["Address", "Telephone", "Year"]   # Year is always 2012
df = df.drop(columns=drop_cols)
print(f"✅ Dropped useless columns: {drop_cols}")

✅ Dropped useless columns: ['Address', 'Telephone', 'Year']


In [15]:
df = df.rename(columns={
    "District Name": "District",
    "Taluk Name"   : "Taluk",
    "Market Name"  : "Market",
})

In [16]:
print(f"\n📋 Missing values per column:")
print(df.isnull().sum())
 
df = df.dropna()
print(f"✅ After dropping NaN rows  →  {df.shape[0]} rows")


📋 Missing values per column:
District     0
Taluk        0
Market       0
Commodity    0
Month        0
Arrival      0
Unit         0
dtype: int64
✅ After dropping NaN rows  →  21422 rows


In [17]:
month_order = {
    "Jan": 1,  "Feb": 2,  "Mar": 3,  "Apr": 4,
    "May": 5,  "Jun": 6,  "Jul": 7,  "Aug": 8,
    "Sep": 9,  "Oct": 10, "Nov": 11, "Dec": 12
}
df["Month_Num"] = df["Month"].map(month_order)
print("✅ Encoded Month → Month_Num (1–12)")

✅ Encoded Month → Month_Num (1–12)


In [18]:
before = len(df)
df = df[df["Arrival"] > 0]
print(f"✅ Removed zero-arrival rows  →  {before - len(df)} rows removed")

✅ Removed zero-arrival rows  →  187 rows removed


In [19]:
Q1 = df["Arrival"].quantile(0.25)
Q3 = df["Arrival"].quantile(0.75)
IQR = Q3 - Q1
upper_cap = Q3 + 3 * IQR          # generous cap — 3×IQR keeps real big markets
df["Arrival"] = df["Arrival"].clip(upper=upper_cap)
print(f"✅ Capped Arrival outliers at {upper_cap:,.0f} (3×IQR rule)")

✅ Capped Arrival outliers at 6,401 (3×IQR rule)


In [20]:
print(f"\n📊 Final cleaned dataset shape  →  {df.shape}")
print(f"\n🔍 Column overview:")
print(df.dtypes)
print(f"\n🔍 Arrival stats after cleaning:")
print(df["Arrival"].describe().round(2))
print(f"\n🔍 Preview:")
print(df.head(8).to_string(index=False))
 


📊 Final cleaned dataset shape  →  (21235, 8)

🔍 Column overview:
District     object
Taluk        object
Market       object
Commodity    object
Month        object
Arrival       int64
Unit         object
Month_Num     int64
dtype: object

🔍 Arrival stats after cleaning:
count    21235.00
mean      1461.39
std       2238.10
min          1.00
25%         55.00
50%        269.00
75%       1641.50
max       6401.00
Name: Arrival, dtype: float64

🔍 Preview:
 District  Taluk Market  Commodity Month  Arrival    Unit  Month_Num
Bagalakot Badami BADAMI      Bajra   Jan      242 Quintal          1
Bagalakot Badami BADAMI       Bull   Jan       65 Numbers          1
Bagalakot Badami BADAMI        Cow   Jan      151 Numbers          1
Bagalakot Badami BADAMI       Goat   Jan      492 Numbers          1
Bagalakot Badami BADAMI  Groundnut   Jan      364 Quintal          1
Bagalakot Badami BADAMI He Baffalo   Jan       41 Numbers          1
Bagalakot Badami BADAMI      Jowar   Jan        4 Quintal 

In [21]:
df.to_csv(OUT_FILE, index=False)
print(f"\n✅ Saved cleaned data  →  {OUT_FILE}")
print("\n▶  Next: run   python step2_eda.py")


✅ Saved cleaned data  →  data/cleaned_data.csv

▶  Next: run   python step2_eda.py
